# Labelling deck tables (sepsis)
Per-cluster shares (mean of per-sample fractions, adult HC / adult sepsis) for real, PBISC K=1,000 and K=4,000, with labeller recall/precision on PBISC-format real half B. Generated cells labelled by the format-aware classifier (R3); R1 = pairwise Harmony + kNN for comparison.

In [ ]:
import pandas as pd, numpy as np, json
O = "/home/claude/work/prop"
T = pd.read_csv(f"{O}/sepsis_sc_truth_props.csv.gz")
L = pd.read_csv(f"{O}/sepsis_mapping_counts.csv.gz")
rc = pd.read_csv(f"{O}/sepsis_confusion_recall.csv")
def psm(d):
    t = d.groupby(["sample", "label"]).n.sum().unstack(fill_value=0); return t.div(t.sum(1), axis=0).mean(0)
out = []
for lv in ["major", "fine"]:
    for G, Tg in [("adult_HC", "adult_HC"), ("adult_sepsis", "adult_sepsis")]:
        tr = T[(T.level == lv) & (T.grp == Tg)].pivot_table(index="sample", columns="label", values="frac", fill_value=0).mean(0)
        row = {"real": tr}
        for K in ["k1000", "k4000"]:
            for rt in ["R1", "R3"]:
                d = L[(L.source == K) & (L.route == rt) & (L.level == lv) & (L.group == G) & (~L.rejected | True)]
                row[f"{K}_{rt}"] = psm(d)
        df = pd.DataFrame(row).fillna(0); df["level"] = lv; df["group"] = G; out.append(df.rename_axis("label").reset_index())
S = pd.concat(out)
q = rc[rc.level.isin(["major", "fine"])].pivot_table(index=["level", "label"], columns="route", values=["recall", "precision", "n_true"])
q.columns = [f"{a}_{b}" for a, b in q.columns]; q = q.reset_index()
W = S.pivot_table(index=["level", "label"], columns="group", values=["real", "k1000_R3", "k4000_R3", "k1000_R1", "k4000_R1"])
W.columns = [f"{a}|{b}" for a, b in W.columns]; W = W.reset_index().merge(q, on=["level", "label"], how="left")
W["judgeable_R3"] = (W.recall_R3 >= .5) & (W.precision_R3 >= .5)
W.to_csv("sepsis_cluster_table.csv", index=False)
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 200)
cols = ["level", "label", "real|adult_HC", "real|adult_sepsis", "k1000_R3|adult_HC", "k1000_R3|adult_sepsis", "k4000_R3|adult_HC", "k4000_R3|adult_sepsis", "k1000_R1|adult_sepsis", "recall_R3", "precision_R3", "recall_R1", "precision_R1", "n_true_R3"]
print((W[cols].set_index(["level", "label"]) * 1).round(3).to_string())


## Summary metrics used on slide 14

In [ ]:
F = W[W.level == "fine"]
J = F[(F.recall_R3 >= .5) & (F.precision_R3 >= .5) & (F["real|adult_sepsis"] >= .003)]
print("judgeable (R3), real sepsis >= 0.3%:", len(J), "/", (F["real|adult_sepsis"] >= .003).sum())
print("judgeable (R1):", ((F.recall_R1 >= .5) & (F.precision_R1 >= .5) & (F["real|adult_sepsis"] >= .003)).sum())
for k in ["k1000_R3", "k4000_R3"]:
    r = J[k + "|adult_sepsis"] / J["real|adult_sepsis"]; ok = (r >= .5) & (r <= 2)
    d = np.sign(J[k + "|adult_sepsis"] - J[k + "|adult_HC"]) == np.sign(J["real|adult_sepsis"] - J["real|adult_HC"])
    print(k, "within 2x:", ok.sum(), "outside:", list(J.label[~ok]), "| direction agrees:", d.sum(), "mismatch:", list(J.label[~d]))